# UrbanTwin • Milestone 5: Space-Time Data Fusion Validation
**Module**: CongestAir Master Dataset (`fused_zone_time`)

This notebook validates the unified space-time master analytical dataset (`fused_zone_time`):
- **Section 1**: Spatial H3 Resolution 7 grid coverage and station-to-zone correspondence
- **Section 2**: AQI and high-resolution Open-Meteo weather temporal alignment (`pd.merge_asof`)
- **Section 3**: Space-time missingness matrix and verification of traffic placeholders (ensuring strictly NULL)
- **Section 4**: Temporal continuity, multi-lag feature readiness, and meteorological correlation analysis

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from spatial.snapping import SpatialEntityResolver
from spatial.grid import generate_h3_grid_pmc_pcmc
from pipelines.fusion import SpaceTimeFusionEngine

# Load fused parquet
parquet_path = project_root / 'data' / 'fused_zone_time.parquet'
df_fused = pd.read_parquet(parquet_path)
print(f'Loaded fused master dataset: {len(df_fused)} records across {df_fused["zone_id"].nunique()} H3 zones.')
df_fused.head()

## Section 1: Spatial H3 Zone Coverage & Station Allocation
We verify that all monitoring stations snap to valid Uber H3 Resolution 7 cells within PMC/PCMC municipal boundaries, respecting the 1.5 km proximity threshold.

In [ ]:
resolver = SpatialEntityResolver(resolution=7)
station_corr = resolver.resolve_all_stations()
print('--- STATION TO H3 ZONE CORRESPONDENCE ---')
print(station_corr[['station_id', 'station_name', 'zone_id', 'distance_m', 'snap_status']])

# Zone aggregation
zone_summary = df_fused.groupby('zone_id').agg(
    records=('station_id', 'count'),
    start_date=('observation_datetime', 'min'),
    end_date=('observation_datetime', 'max'),
    mean_aqi=('aqi', 'mean'),
    mean_temp=('temperature_2m', 'mean')
).reset_index()
zone_summary

## Section 2: AQI + Weather Temporal Alignment Inspection
Check temporal alignment between daily AQI observations and hourly Open-Meteo meteorological parameters (temperature, humidity, precipitation, wind, surface pressure, boundary layer height).

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))

zone_sample = df_fused['zone_id'].iloc[0]
df_zone = df_fused[df_fused['zone_id'] == zone_sample].sort_values('observation_datetime')

color = 'tab:red'
ax1.set_xlabel('Observation Datetime')
ax1.set_ylabel('AQI (Daily Mean)', color=color)
ax1.plot(df_zone['observation_datetime'], df_zone['aqi'], color=color, alpha=0.7, label='AQI')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:blue'
ax2.set_ylabel('Temperature 2m (C)', color=color)
ax2.plot(df_zone['observation_datetime'], df_zone['temperature_2m'], color=color, alpha=0.5, linestyle='--', label='Temp (C)')
ax2.tick_params(axis='y', labelcolor=color)

plt.title(f'AQI and Aligned Weather Over Time (Zone {zone_sample})')
fig.tight_layout()
plt.show()

## Section 3: Space-Time Missingness & Traffic Placeholder Verification
Verify that:
1. Traffic features are strictly `NULL` / `NaN` as traffic collection is not yet active.
2. Missingness flags (`aqi_missing`, `weather_missing`, `traffic_missing`) correctly flag observation completeness.

In [ ]:
traffic_cols = ['traffic_speed', 'traffic_congestion', 'traffic_travel_time', 'traffic_confidence']
traffic_null_check = {col: df_fused[col].isna().all() for col in traffic_cols}
print('Traffic columns strictly NULL verification:')
for col, is_all_null in traffic_null_check.items():
    status = 'PASS (All NULL)' if is_all_null else 'FAIL (Contains values)'
    print(f'  - {col}: {status}')

# Missingness breakdown
flags_summary = pd.DataFrame({
    'flag': ['weather_missing', 'aqi_missing', 'traffic_missing'],
    'true_count': [df_fused['weather_missing'].sum(), df_fused['aqi_missing'].sum(), df_fused['traffic_missing'].sum()],
    'true_percentage': [
        df_fused['weather_missing'].mean() * 100,
        df_fused['aqi_missing'].mean() * 100,
        df_fused['traffic_missing'].mean() * 100
    ]
})
flags_summary

## Section 4: Meteorological Correlation & Lag Readiness
Validate that physical relationships between boundary layer height, temperature, wind speed, and pollutant concentrations hold prior to causal inference.

In [ ]:
analysis_cols = ['aqi', 'pm25', 'pm10', 'no2', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'boundary_layer_height', 'surface_pressure']
corr_matrix = df_fused[analysis_cols].corr()
corr_matrix.round(3)

print('Validation complete. Master table fused_zone_time is verified and ready for downstream forecasting and causal Granger testing.')